# ML-07 — Baseline Action Score and Top-20 Review

This baseline prioritizes content pages for review using only signals available in the starter snapshot. The rule is intentionally simple, readable, and fixed before model work.

**Decision:** which pages should a content/SEO team review first?

**Important:** `trend_direction`, `trend_pct`, and `is_declining_label` are not used to build the score. The label is used only for retrospective evaluation.


## 1. Signal checks and my rule

### Signal 1 — Freshness / staleness
**Hypothesis:** older pages since their last update are more likely to need review.

I will compare the observed declining-label rate across freshness buckets.  
**Flag link:** this is the signal behind FlyRank's refresh/staleness flags.

### Signal 2 — Search visibility / volume
**Hypothesis:** pages with more impressions are higher-value review opportunities than pages with little search visibility.

I will compare the observed declining-label rate across impression buckets.  
**Flag link:** this is the volume signal behind FlyRank's quick-win logic.

The verdicts below are data checks, not assumptions.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)

# Label is evaluation-only; never use it to construct the score.
target = "is_declining_label"
df[target] = (df["trend_direction"] == "down").astype(int)

# Signal 1: freshness / staleness
fresh_bins = [-np.inf, 30, 90, 180, 365, np.inf]
fresh_labels = ["0-30", "31-90", "91-180", "181-365", "365+"]
df["freshness_bucket"] = pd.cut(
    df["days_since_last_update"], bins=fresh_bins,
    labels=fresh_labels, right=True, include_lowest=True
)

fresh_check = (
    df.groupby("freshness_bucket", observed=False)[target]
      .agg(n="size", declining_rate="mean")
      .reset_index()
)
print("\nSignal 1 — freshness bucket table (n printed):")
print(fresh_check.to_string(index=False))

# Signal 2: search visibility / volume
vol_bins = [-np.inf, 0, 100, 1000, 3000, 30000, np.inf]
vol_labels = ["0", "1-100", "101-1k", "1k-3k", "3k-30k", "30k+"]
df["volume_bucket"] = pd.cut(
    df["impressions_90d"], bins=vol_bins,
    labels=vol_labels, right=True, include_lowest=True
)

volume_check = (
    df.groupby("volume_bucket", observed=False)[target]
      .agg(n="size", declining_rate="mean")
      .reset_index()
)
print("\nSignal 2 — impression bucket table (n printed):")
print(volume_check.to_string(index=False))

fresh_rate = fresh_check.set_index("freshness_bucket")["declining_rate"]
fresh_delta = fresh_rate.get("181-365", np.nan) - fresh_rate.get("0-30", np.nan)

vol_rate = volume_check.set_index("volume_bucket")["declining_rate"]
vol_delta = vol_rate.get("3k-30k", np.nan) - vol_rate.get("1-100", np.nan)

fresh_verdict = "CONFIRMED" if fresh_delta > 0 else "OPPOSITE" if fresh_delta < 0 else "MIXED"
volume_verdict = "CONFIRMED" if vol_delta > 0 else "OPPOSITE" if vol_delta < 0 else "MIXED"

print(f"\nSignal 1 verdict: {fresh_verdict}")
print("Reason: compares the observed declining rate for stale pages with a younger reference bucket.")
print(f"Signal 2 verdict: {volume_verdict}")
print("Reason: compares observed declining rate for higher-visibility pages with a lower-volume reference bucket.")


### Rule in plain words

**Review a page first when it is both stale (at least 180 days since its last update) and visibly receiving search impressions (at least 3,000 impressions in 90 days).**

The score is the page's 90-day impressions when both conditions are true; otherwise the score is zero.

This is a deliberately transparent baseline. It does not fit weights and it does not use the future-derived label.


## 2. Build the ranked queue

The queue contains every content item, one row per page.  
Columns: `content_id`, `score`, `reason_code`, `action_label`.


In [ ]:
# Transparent rule: no fitted weights, no label-derived inputs.
stale = df["days_since_last_update"].fillna(0) >= 180
visible = df["impressions_90d"].fillna(0) >= 3000

df["score"] = np.where(stale & visible, df["impressions_90d"].fillna(0), 0.0)

df["reason_code"] = np.where(
    stale & visible,
    "stale_but_visible",
    "not_flagged"
)

df["action_label"] = np.where(
    stale & visible,
    "refresh_review",
    "monitor"
)

queue = (
    df[["content_id", "score", "reason_code", "action_label"]]
    .sort_values(["score", "content_id"], ascending=[False, True])
    .reset_index(drop=True)
)

out_dir = Path("../../work/outputs")
if not out_dir.exists():
    out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "baseline_action_score.csv"
queue.to_csv(out_path, index=False)

print("Rows ranked:", len(queue))
print("Flagged for refresh review:", int((queue["action_label"] == "refresh_review").sum()))
print("Queue written to:", out_path)
print("\nTop 10:")
print(queue.head(10).to_string(index=False))


## Baseline evaluation

Precision@K is reported only as a retrospective check against the observed label. The label is **not** an input to the rule.


In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df[target].mean()
print(f"Base rate (observed declining label): {base_rate:.3f}")

for k in [20, 50]:
    p_at_k = precision_at_k(df["score"], df[target], k)
    print(f"Precision@{k}: {p_at_k:.3f}")


## 3. Top-20 review

For each selected page: state the action, why it is there, and what would make the pick wrong.  
This review is intentionally skeptical: a high score means high visibility among stale pages, not proof that a refresh will help.


In [ ]:
top20 = queue.head(20).merge(
    df[["content_id", "days_since_last_update", "impressions_90d", "avg_position"]],
    on="content_id", how="left"
)

for i, row in top20.iterrows():
    wrong = (
        "Wrong if the page is already accurate and current, the traffic is not commercially useful, "
        "or the apparent visibility is driven by a measurement artifact rather than a real opportunity."
    )
    print(
        f"{i+1}. {row['content_id']} — action: {row['action_label']}; "
        f"why: {row['reason_code']} (age={row['days_since_last_update']:.0f}d, "
        f"impressions_90d={row['impressions_90d']:.0f}, score={row['score']:.0f}); "
        f"what would make it wrong: {wrong}"
    )


## 4. Weak picks + leakage check

The weakest picks are useful because they show where a transparent rule can be too blunt.


In [ ]:
weak = queue[queue["score"] > 0].tail(5)
print("Weak non-zero picks:")
print(weak.to_string(index=False))

score_inputs = {"days_since_last_update", "impressions_90d"}
forbidden = {"trend_direction", "trend_pct", "is_declining_label"}
assert score_inputs.isdisjoint(forbidden)

print("\nLeakage check: PASS")
print("Rule inputs:", sorted(score_inputs))
print("Forbidden label-source fields were not used in score construction.")


## 5. Self-check

- [x] Two signal checks with visible bucket tables and `n`.
- [x] At least one signal is directly linked to a FlyRank flag: staleness/refresh and volume/quick-win.
- [x] One transparent rule with a score, one reason code, and an action label.
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv`.
- [x] Top-20 review includes action, reason, and what would make each pick wrong.
- [x] No future-window or label-derived inputs in the rule.
- [x] Precision@K and base rate are shown as retrospective evaluation only.
